### The objective of this notebook is to apply hierarchical reconciliation through MinTrace (Minimum Trace) method to the base forecasts, so as to ensure that the forecasts sum coherently across the hierarchy (Route -> Airport -> Total) 

In [2]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoARIMA, AutoETS

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace
from hierarchicalforecast.utils import aggregate

In [3]:
# Load the full processed dataset
df_full = pd.read_csv('../data/processed/hierarchy_ready.csv')
df_full['ds'] = pd.to_datetime(df_full['ds'])

In [4]:
# Dropping the 4 missing series
to_drop = {
    'TOTAL/MUMBAI/MUMBAI-RAJKOT',
    'TOTAL/DELHI/DELHI-RAJKOT',
    'TOTAL/DEHRA DUN/DEHRA DUN-DELHI',
    'TOTAL/DEHRA DUN'
}

df_full = df_full[~df_full['unique_id'].isin(to_drop)].copy()

In [5]:
# Train Test Split (Same as that used for forecasting)

split_date = pd.Timestamp('2025-03-01')
train = df_full[df_full['ds'] < split_date].copy()
test = df_full[df_full['ds'] >= split_date].copy()

In [6]:
# Loading the baseforecasts

base_forecasts = pd.read_csv('../outputs/base_forecasts.csv')
base_forecasts['ds'] = pd.to_datetime(base_forecasts['ds'])

In [8]:
print('Full data: ',df_full.shape, ' | Series: ',df_full['unique_id'].nunique())
print("Train: ",train.shape, " | Test: ",test.shape)
print('Base Forecasts: ',base_forecasts.shape)
print('Models in base forecasts: ',[c for c in base_forecasts.columns if c not in ['ds','unique_id']])

Full data:  (26677, 3)  | Series:  228
Train:  (23941, 3)  | Test:  (2736, 3)
Base Forecasts:  (2736, 6)
Models in base forecasts:  ['Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']


In [9]:
# Building the hierarchy structure

# Preparing data for the aggregate function

# Filter for route series

routes_only = df_full[df_full['unique_id'].str.count('/') == 2].copy()

# Extract the origin and destination for each unique id

routes_only['origin'] = routes_only['unique_id'].str.split('/').str[1]

routes_only['route_label'] = routes_only['unique_id'].str.split('/').str[2]

print("Route rows: ", routes_only.shape)

print("Unique routes: ",routes_only['route_label'].nunique())

print('Unique origin airports: ',routes_only['origin'].nunique())

print(routes_only[['unique_id','origin','route_label','ds','y']].head())


Route rows:  (22955, 5)
Unique routes:  197
Unique origin airports:  30
                            unique_id    origin  ...         ds        y
213  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-05-01  11338.0
214  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-06-01  10786.0
215  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-07-01  11355.0
216  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-08-01  10927.0
217  TOTAL/AGARTALA/AGARTALA-GUWAHATI  AGARTALA  ... 2019-09-01   9271.0

[5 rows x 5 columns]


In [19]:
# Define the hierarchy specification
routes_for_agg = routes_only.drop(columns = ['unique_id']).copy()
routes_for_agg['total'] = 'TOTAL'
spec = [
    ['total'],
    ['total','origin'],
    ['total','origin','route_label']
]

# Build the hierarchy
Y_df, S_df, tags = aggregate(routes_for_agg, spec)

print('Y_df (long-format reconstructed hierarchy): ', Y_df.shape)
print('S_df (the S matrix): ',S_df.shape)
print('Tags (groups of series at each level): ',{k:len(v) for k, v in tags.items()})
print('S matrix preview: ')
print(S_df.head())

Y_df (long-format reconstructed hierarchy):  (26677, 3)
S_df (the S matrix):  (228, 198)
Tags (groups of series at each level):  {'total': 1, 'total/origin': 30, 'total/origin/route_label': 197}
S matrix preview: 
         unique_id  ...  TOTAL/NAGPUR/NAGPUR-PUNE
0            TOTAL  ...                       1.0
1   TOTAL/AGARTALA  ...                       0.0
2  TOTAL/AHMEDABAD  ...                       0.0
3     TOTAL/AIZAWL  ...                       0.0
4   TOTAL/AMRITSAR  ...                       0.0

[5 rows x 198 columns]


In [20]:
# Compute in-sample residuals

Y_df['ds'] = pd.to_datetime(Y_df['ds'])

# Train test split on Y_df
split_date = pd.Timestamp('2025-03-01')
Y_train = Y_df[Y_df['ds'] < split_date].copy()
Y_test = Y_df[Y_df['ds']>=split_date].copy()

# Refit the same 4 models on Y_train
models = [
    Naive(),
    SeasonalNaive(season_length = 12),
    AutoARIMA(season_length = 12),
    AutoETS(season_length = 12)
]

sf = StatsForecast(models = models, freq = 'MS', n_jobs = -1, verbose = True)
sf.fit(df = Y_train)

# Generate forecasts with fitted values = True to also get in-sample fitted values
fcst_with_insample = sf.forecast(df = Y_train, h = 12, fitted = True)

print('Forecast Shape: ',fcst_with_insample.shape)
print('Columns: ',fcst_with_insample.columns.tolist())

Forecast:   0%|          | 0/228 [Elapsed: 00:00]

Forecast Shape:  (2736, 6)
Columns:  ['unique_id', 'ds', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']


In [27]:
# Get in-sample fitted values (training period predictions per model)

fitted_vals = sf.forecast_fitted_values()

print('Fitted values shape: ',fitted_vals.shape)
print('Columns: ',fitted_vals.columns.tolist())
print('Sample: ')
print(fitted_vals.head())

Fitted values shape:  (23941, 7)
Columns:  ['unique_id', 'ds', 'y', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS']
Sample: 
  unique_id         ds          y  ...  SeasonalNaive     AutoARIMA       AutoETS
0     TOTAL 2015-04-01  2329936.0  ...            NaN  2.327606e+06  2.326358e+06
1     TOTAL 2015-05-01  2512608.0  ...            NaN  2.334287e+06  2.329936e+06
2     TOTAL 2015-06-01  2296222.0  ...            NaN  2.550999e+06  2.512590e+06
3     TOTAL 2015-07-01  2401175.0  ...            NaN  2.239675e+06  2.296244e+06
4     TOTAL 2015-08-01  2352577.0  ...            NaN  2.437065e+06  2.401165e+06

[5 rows x 7 columns]


In [28]:
# Apply MinTrace Reconciliation

# Set up reconciliation engine with two MinTrace variants
reconcilers = [
    MinTrace(method = 'ols'),
    MinTrace(method = 'mint_shrink')
]

hrec = HierarchicalReconciliation(reconcilers = reconcilers)

# Apply the reconciliation
Y_rec_df = hrec.reconcile(
    Y_hat_df = fcst_with_insample,
    Y_df = fitted_vals,
    S_df= S_df,
    tags = tags
)

print('Reconciled Forecasts shape: ',Y_rec_df.shape)
print('Columns: ',Y_rec_df.columns.tolist())
print('Sample: ')
print(Y_rec_df.head())

Reconciled Forecasts shape:  (2736, 14)
Columns:  ['unique_id', 'ds', 'Naive', 'SeasonalNaive', 'AutoARIMA', 'AutoETS', 'Naive/MinTrace_method-ols', 'SeasonalNaive/MinTrace_method-ols', 'AutoARIMA/MinTrace_method-ols', 'AutoETS/MinTrace_method-ols', 'Naive/MinTrace_method-mint_shrink', 'SeasonalNaive/MinTrace_method-mint_shrink', 'AutoARIMA/MinTrace_method-mint_shrink', 'AutoETS/MinTrace_method-mint_shrink']
Sample: 
  unique_id  ... AutoETS/MinTrace_method-mint_shrink
0     TOTAL  ...                        5.777137e+06
1     TOTAL  ...                        5.393259e+06
2     TOTAL  ...                        5.013512e+06
3     TOTAL  ...                        4.989821e+06
4     TOTAL  ...                        5.067028e+06

[5 rows x 14 columns]


In [ ]:
# Coherence Sanity check
